# FINS5557 project notebook

Our one notebook for exploring the data and developing the model. Code that works and gets reused (loading, features, training, SHAP, fairness, PSI) is moved into `src/`, and this notebook imports it, so the app and the notebook always use the same steps.

| Section | Status |
|---|---|
| 0. Setup | done |
| 1. Data: exploration and checks | done |
| 2. Features | done |
| 3. Models: baseline vs random forest vs XGBoost | done |
| 4. Calibration and threshold | done |
| 5. Explainability (SHAP) and fairness | to do |

See `docs/BUILD_PLAN.md` for the full plan.

**Before you run:** activate `.venv`, pick it as this notebook's kernel (top right, **Select Kernel**), then run top to bottom. Clear outputs before committing.

## 0. Setup
### 0.1 Install packages
Installs `requirements-dev.txt` (which includes `requirements.txt`). Skip if you've already installed them.

In [ ]:
%pip install -q -r ../requirements-dev.txt

### 0.2 Imports and package versions
Prints each version, so a broken setup shows up here rather than half-way through.

`autoreload` picks up changes to `src/` files. If you still get an import error after code changes, use **Restart** in the notebook toolbar, then **Run All**.

In [ ]:
# Reload src/ modules automatically when their files change, so edits to src/
# show up without restarting the kernel.
%load_ext autoreload
%autoreload 2

import sys
import warnings
from importlib.metadata import version
from pathlib import Path

# Make `src` importable whether the notebook runs from the project root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src.config import GOOGLE_API_KEY, LLM_BACKEND, PROTECTED, RANDOM_STATE, TARGET
from src.data import BILL_AMT, PAY_STATUS, clean, features_target, load_raw, split

warnings.filterwarnings("ignore", category=FutureWarning)
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:.3f}".format)

print("Python", sys.version.split()[0])
for pkg in ["pandas", "numpy", "scikit-learn", "xgboost", "shap", "streamlit", "python-dotenv", "joblib", "ucimlrepo"]:
    print(f"{pkg:15s} {version(pkg)}")

### 0.3 Settings check
Reports only whether an API key was found, never the key itself.

In [ ]:
print("Random seed:", RANDOM_STATE)
print("LLM backend:", LLM_BACKEND)
print("GOOGLE_API_KEY found:", bool(GOOGLE_API_KEY))

## 1. Data: exploration and checks

**Findings** (this becomes the Data section of the report):
- **Default rate:** 22.1% (6,636 of 30,000 customers default next month), identical (22.1%) in the train, validation and test splits. A model that always predicts "no default" would score 77.9% accuracy, so we judge models on AUC, KS and Brier score, and use class weights.
- **Strongest early signal:** last month's repayment status `PAY_0` (correlation 0.33 with default; the other `PAY_*` months follow, 0.26 down to 0.19). Customers 1+ months late default at 50.3%, vs 13.8% for everyone else; at 2+ months late it is 69.6%. Credit limit also matters: the lowest limit quintile (≤ NT$50k) defaults at 31.8%, the highest (> NT$270k) at 13.8%, consistent with lenders giving riskier customers lower limits.
- **Data issues and how we handle them:**
  - No missing values; schema checked by `validate_schema()`.
  - Undocumented codes: EDUCATION 0/5/6 (345 rows) → 4 "other", MARRIAGE 0 (54 rows) → 3 "other", done by `clean()`.
  - `PAY_*` values -2 and 0 aren't in the UCI documentation but cover 58% of `PAY_0`; kept as-is (read as "no spending" and "paid the minimum").
  - 590 customers have a negative last bill (credit balance) and 2,598 have a bill ≤ 0, so ratio features in Step 2 need safe division.
  - 35 exact duplicate rows (56 if the target is ignored, so some identical profiles have different outcomes). Kept, since there's no customer ID to prove they're the same person.
  - Amounts are highly skewed (`BILL_AMT1` from -165,580 to 964,511), so logistic regression needs scaling.
- **Fairness preview (raw default rates, before any model):**
  - Men default more than women (24.2% vs 20.8%).
  - Default rises as education falls (graduate school 19.2%, university 23.7%, high school 25.2%); the small "other" group is 7.1%.
  - Married 23.5% vs single 20.9%.
  - By age it is U-shaped: lowest at 30–39 (20.3%), highest at 60+ (28.3%, only 339 people).
  - Because these groups already differ in raw risk, a model that never sees SEX, MARRIAGE or AGE can still give them different approval rates through correlated inputs. Step 10 checks this with the disparate-impact ratio.

### 1.1 Load the data
The first run downloads it from UCI and saves `data/raw/uci_credit_default.csv` (gitignored). Later runs read the local file.

In [ ]:
raw = load_raw()
print(raw.shape)  # expect (30000, 24)
raw.head()

### 1.2 Target: how many customers default?
About 22% is expected. This imbalance is why the models use class weights, and why accuracy alone would be misleading (predicting "no default" for everyone scores 78%).

In [ ]:
raw[TARGET].value_counts(normalize=True).rename("share")

### 1.3 Data quality checks

In [ ]:
print("Missing values:", int(raw.isna().sum().sum()))
print("Duplicate rows:", int(raw.duplicated().sum()))
print("Customers with a negative bill (credit balance) last month:", int((raw["BILL_AMT1"] < 0).sum()))
print()
print("EDUCATION codes:", raw["EDUCATION"].value_counts().sort_index().to_dict())
print("MARRIAGE codes: ", raw["MARRIAGE"].value_counts().sort_index().to_dict())

EDUCATION 0, 5, 6 and MARRIAGE 0 are not in the UCI documentation, so `clean()` groups them into "other". Duplicates are kept: with no customer ID we can't tell whether they are the same person.

In [ ]:
df = clean(raw)
print("EDUCATION after cleaning:", df["EDUCATION"].value_counts().sort_index().to_dict())
print("MARRIAGE after cleaning: ", df["MARRIAGE"].value_counts().sort_index().to_dict())

### 1.4 Repayment status vs default
`PAY_0` is last month's status: -1 paid on time, 1 to 9 months late (-2 and 0 are usually read as no spending and paying the minimum). Expect this to be the strongest predictor.

In [ ]:
(df.groupby("PAY_0")[TARGET]
   .agg(customers="size", default_rate="mean")
   .round(3))

### 1.5 Amounts
Credit limits and bills are in NT dollars and very skewed, which matters for scaling in logistic regression (tree models don't mind).

In [ ]:
df[["LIMIT_BAL", "AGE", *BILL_AMT[:2], "PAY_AMT1"]].describe().T

In [ ]:
# Default rate by credit-limit quintile: do customers with lower limits default more?
df.groupby(pd.qcut(df["LIMIT_BAL"], 5), observed=True)[TARGET].mean().rename("default_rate")

### 1.6 Fairness preview
These attributes are **not** model inputs, but we audit the model's decisions by them later. Here are the raw default rates by group as a baseline.

In [ ]:
age_band = pd.cut(df["AGE"], [20, 30, 40, 50, 60, 80], right=False)
for name, group in [("SEX (1 = male, 2 = female)", df["SEX"]), ("EDUCATION", df["EDUCATION"]),
                    ("MARRIAGE", df["MARRIAGE"]), ("Age band", age_band)]:
    print(name)
    print(df.groupby(group, observed=True)[TARGET].agg(customers="size", default_rate="mean").round(3), "\n")

### 1.7 Train / validation / test split
Stratified 60/20/20, done before any model fitting. The default rate should be the same in all three parts.

In [ ]:
train, val, test = split(df)
pd.DataFrame({
    "rows": [len(train), len(val), len(test)],
    "default_rate": [train[TARGET].mean(), val[TARGET].mean(), test[TARGET].mean()],
}, index=["train", "validation", "test"])

In [ ]:
X_train, y_train = features_target(train)
print(X_train.shape[1], "model inputs; protected attributes removed:", PROTECTED)
list(X_train.columns)

## 2. Features

New features built by `add_features()` in `src/features.py`, each a standard credit-risk idea:

| Feature | Definition | Credit-risk meaning |
|---|---|---|
| `utilisation` | `BILL_AMT1 / LIMIT_BAL` | How much of the limit is used; high use signals stress |
| `payment_ratio` | `PAY_AMT1 / BILL_AMT2`, 1.0 if the bill is ≤ 0, capped at 1 | Paying only a small share means revolving debt. Uses `BILL_AMT2` because September's payment settles the August statement |
| `months_late_6m` | number of `PAY_*` > 0 | How often the customer was in arrears |
| `max_months_late` | largest `PAY_*`, floored at 0 | How bad the worst arrears were |
| `avg_utilisation_6m` | mean of `BILL_AMTi / LIMIT_BAL` | Utilisation smoothed over 6 months |
| `bill_trend` | `BILL_AMT1 - BILL_AMT6` | Positive = balance growing, negative = paying it down |

Divisions are safe: a zero or negative bill or limit never produces NaN or infinity (590 customers have a credit balance). Everything here uses the **training split only**, so validation and test stay unseen.

**Findings** (training split, 18,000 rows):
- **`months_late_6m` is the strongest single signal** (correlation 0.40 with default, above `PAY_0` at 0.33). Default rises steadily from 11.6% for customers never late to 68.7% for those late in all 6 months. `max_months_late` is close behind (0.37).
- **Utilisation is U-shaped, not linear.** Default is 15.1% at low utilisation (1–13% of the limit) and 27–28% above 52%, but customers with near-zero use also default more (21.4%), probably inactive or new accounts. `avg_utilisation_6m` shows a clearer pattern (14.4% → 32.5% from the second to the top quintile).
- **Low payers are riskier:** customers paying under 4.1% of their previous bill default at 32.0%, vs 16.8% for those paying more than 23% (correlation -0.11). 33.1% of customers get the full value of 1.0 (bill paid in full, or nothing owed). Matching the payment to the bill it actually pays (`BILL_AMT2`) gave a stronger signal than using the same month's bill (correlation -0.07).
- **`bill_trend` is weak** (correlation -0.03) and non-linear; we keep it and let the models decide.
- These curved relationships are a reason to expect tree models (random forest, XGBoost) to beat logistic regression, which we test in section 3.

In [ ]:
from src.config import ENGINEERED_FEATURES
from src.features import add_features, build_preprocessor

train_f = add_features(train)
train_f[ENGINEERED_FEATURES].describe().T

### 2.1 Each feature against the default rate
Correlation gives a quick ranking; the quintile tables show the shape of each relationship (a correlation near 0 can still hide a U-shape).

In [ ]:
train_f[[*ENGINEERED_FEATURES, TARGET]].corr()[TARGET].drop(TARGET).sort_values(key=abs, ascending=False).rename("correlation")

In [ ]:
for col in ["utilisation", "avg_utilisation_6m", "payment_ratio", "bill_trend"]:
    bins = pd.qcut(train_f[col], 5, duplicates="drop")  # payment_ratio has ties at 1.0, so fewer bins
    print(col)
    print(train_f.groupby(bins, observed=True)[TARGET].agg(customers="size", default_rate="mean").round(3))
    print()

train_f.groupby("months_late_6m")[TARGET].agg(customers="size", default_rate="mean").round(3)

### 2.2 Preprocessing pipeline
`build_preprocessor()` scales the numeric columns (needed for logistic regression, harmless for trees) and one-hot encodes `EDUCATION`. It is fitted on the training split only and then applied unchanged to validation, so both get the same columns. Protected attributes are dropped. Pattern adapted from Code Bank W06-04.

In [ ]:
pre = build_preprocessor()
X_train_ready = pre.fit_transform(train_f)
X_val_ready = pre.transform(add_features(val))
print("train:", X_train_ready.shape, " validation:", X_val_ready.shape)
list(pre.get_feature_names_out())

## 3. Models: baseline vs random forest vs XGBoost

Training lives in `src/train.py`. Run it once from the project root (about a minute):

```bash
python -m src.train
```

It tunes each model with 5-fold stratified cross-validation on the **training split only**, scores all three on the **validation split**, applies the model-choice rule and saves `models/model.joblib`, `models/metadata.json` and `reports/model_comparison.csv`. This section reads those files instead of retraining. The test split is still untouched.

| Model | Why it's in the comparison | Imbalance handling |
|---|---|---|
| Logistic regression | Traditional scorecard baseline (Code Bank W01-06, W05-02): linear, transparent, easy to defend to a regulator | `class_weight="balanced"` |
| Random forest | Many decision trees averaged: captures non-linear patterns and interactions | `class_weight="balanced"` |
| XGBoost | Boosted trees, usually the strongest on tabular credit data (Code Bank W02-06, W05-03) | `scale_pos_weight` = non-defaulters / defaulters ≈ 3.5 |

Every model is one pipeline (feature engineering → scaling / one-hot → model), so features are fitted inside each CV fold and the saved model takes raw applicant columns.

**Model-choice rule** (`MIN_AUC_GAIN` in `src/config.py`): take the highest validation AUC, but a tree model has to beat logistic regression by at least 0.01 AUC to justify the loss of transparency.

**Findings** (validation split, 6,000 customers):
- **Both tree models beat the baseline:** random forest AUC 0.784 and XGBoost 0.783, vs 0.763 for logistic regression. That is a Gini of 0.567 vs 0.525 and a KS of 0.44 vs 0.41, so the tree models separate defaulters from non-defaulters better, consistent with the curved relationships found in section 2.
- **Chosen: random forest.** It clears the 0.01 margin (+0.021 AUC). XGBoost is effectively tied (0.001 lower AUC, slightly higher KS and balanced accuracy), so the choice between the two tree models is not material; either would be defensible.
- **No overfitting:** cross-validation AUC on train matches validation AUC for every model (e.g. random forest 0.784 vs 0.784).
- **Probabilities are not yet PDs:** class weighting pushes predicted probabilities upwards (Brier score 0.177–0.189). Step 4 calibrates them so a "20% PD" really means about 1 in 5 default.

In [ ]:
import json

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

from src.config import METADATA_PATH, REPORTS_DIR

comparison = pd.read_csv(REPORTS_DIR / "model_comparison.csv", index_col="model")
metadata = json.loads(METADATA_PATH.read_text())
print("Chosen:", metadata["chosen_model"], "|", metadata["reason"])
print("Best settings:", metadata["best_params"])
comparison

### 3.1 ROC curves on the validation split
Each curve shows the trade-off between catching defaulters (true positive rate) and wrongly flagging good customers (false positive rate) across all cut-offs. A higher curve means better ranking; the dashed line is a coin flip (AUC 0.5).

In [ ]:
val_preds = pd.read_csv(REPORTS_DIR / "validation_predictions.csv")

fig, ax = plt.subplots(figsize=(6, 5))
for name in comparison.index:
    fpr, tpr, _ = roc_curve(val_preds[TARGET], val_preds[name])
    ax.plot(fpr, tpr, label=f"{name} (AUC {comparison.loc[name, 'auc']:.3f})")
ax.plot([0, 1], [0, 1], "--", color="grey", label="random (AUC 0.500)")
ax.set(xlabel="False positive rate (good customers flagged)",
       ylabel="True positive rate (defaulters caught)", title="ROC curves: validation split")
ax.legend(loc="lower right")
plt.show()

## 4. Calibration and threshold

Code in `src/threshold.py`, run as part of `python -m src.train`. Three steps turn a model score into a credit decision:

1. **Calibrate.** Class weighting makes the random forest rank customers well but overstate risk (its average score is well above the 22% default rate). A calibrator fitted on the validation split maps scores to **PDs** that match observed default rates. Sigmoid (Platt) and isotonic calibration are compared on out-of-fold Brier score, so neither is judged on rows it was fitted on.
2. **Choose the cut-off from the lender's economics**, not 0.5. Approving a customer earns `MARGIN × EAD` if they repay and loses `LGD × EAD` if they default, so the expected loss is `PD × LGD × EAD` (the Basel formula). Assumptions in `src/config.py`: LGD = 60%, net margin = 12%, EAD = credit limit. The break-even PD is `margin / (margin + LGD)` = 16.7%. The cut-off is the PD that maximises **realised** profit on the validation split.
3. **Decide:** approve below `cut-off - 2 points`, decline above `cut-off + 2 points`, refer the band in between to an analyst (`REFER_BAND` in `src/config.py`).

The test split is then scored **once**, at the chosen cut-off.

**Findings:**
- **Calibration works:** sigmoid was chosen (validation Brier 0.135 vs 0.177 uncalibrated; isotonic 0.135, a near tie). The mean test PD is 22.4% against an actual default rate of 22.1%.
- **The cut-off is a PD of 14.0%,** a little below the 16.7% break-even. The realised-profit curve is flat between about 12% and 20% (NT$38–39m on the validation split), so anywhere in that range is close to optimal. The expected-profit curve peaks exactly at the break-even PD, as theory predicts for calibrated PDs.
- **On the test set (6,000 customers, used once):** AUC 0.786, Gini 0.573, KS 0.426, in line with validation, so the model generalises. The model approves 49.3% of applicants, and the default rate among them falls to **8.6%** from 22.1% overall. It catches 1,072 of 1,327 defaulters (81%), at the cost of declining 1,969 customers who would have repaid.
- **Refer band of ±2 points:** the median PD (13.9%) sits right at the cut-off, so the band width matters a lot: ±5 points (the first idea in the plan) would send 37% of applicants to an analyst, which no lender could staff. ±2 points refers about 15% (see 4.3). On the test set the decisions split into **41.6% approve, 14.1% refer, 44.3% decline**.

In [ ]:
from sklearn.calibration import calibration_curve

from src.threshold import break_even_pd, decide

print("Calibration method:", metadata["calibration"]["method"])
print("Validation Brier:", metadata["calibration"]["validation_brier"])
print("Assumptions:", metadata["assumptions"])
print(f"Cut-off: decline at PD >= {metadata['threshold']:.3f}, refer band +/- {metadata['refer_band']}")

### 4.1 Reliability curve
Customers grouped into 10 bins by predicted risk: average prediction (x) against the share that actually defaulted (y). A calibrated model sits on the diagonal. Validation split, out-of-fold calibrated PDs.

In [ ]:
chosen = metadata["chosen_model"]
fig, ax = plt.subplots(figsize=(6, 5))
for label, scores in [(f"{chosen}, uncalibrated", val_preds[chosen]), ("calibrated PD", val_preds["calibrated_pd"])]:
    observed, predicted = calibration_curve(val_preds[TARGET], scores, n_bins=10, strategy="quantile")
    ax.plot(predicted, observed, "o-", label=label)
ax.plot([0, 1], [0, 1], "--", color="grey", label="perfect calibration")
ax.set(xlabel="Predicted probability of default", ylabel="Observed default rate",
       title="Reliability curve: validation split")
ax.legend()
plt.show()

### 4.2 Profit curve
Profit on the validation split if everyone below each cut-off is approved. **Realised** uses what actually happened; **expected** uses the PDs. Too low a cut-off turns away profitable customers, too high a cut-off approves loss-making ones.

In [ ]:
curve = pd.read_csv(REPORTS_DIR / "profit_curve.csv")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(curve["threshold"], curve["realised_profit"] / 1e6, label="realised profit")
ax.plot(curve["threshold"], curve["expected_profit"] / 1e6, label="expected profit (from PDs)")
ax.axvline(metadata["threshold"], color="black", label=f"chosen cut-off ({metadata['threshold']:.3f})")
ax.axvline(break_even_pd(), color="grey", linestyle="--", label=f"break-even PD ({break_even_pd():.3f})")
ax.axvspan(metadata["threshold"] - metadata["refer_band"], metadata["threshold"] + metadata["refer_band"],
           color="orange", alpha=0.15, label="refer band")
ax.set(xlabel="PD cut-off (approve below)", ylabel="Profit, NT$ million", title="Profit by cut-off: validation split")
ax.legend()
plt.show()

curve.set_index("threshold").loc[[0.10, 0.12, 0.14, 0.16, 0.20, 0.30, 0.50]]

### 4.3 How wide should the refer band be?
Share of validation applicants sent to an analyst for different band widths around the cut-off.

In [ ]:
pds = val_preds["calibrated_pd"]
pd.DataFrame({
    band: pd.Series(decide(pds, metadata["threshold"], band)).value_counts(normalize=True)
    for band in [0.01, 0.02, 0.03, 0.05]
}).T.rename_axis("refer band (+/-)").round(3)[["approve", "refer", "decline"]]

### 4.4 Final test-set results (reported once)
The confusion matrix treats "decline" as predicting default (refer band ignored).

In [ ]:
test_metrics = json.loads((REPORTS_DIR / "test_metrics.json").read_text())
cm = test_metrics["confusion_matrix"]
print(pd.DataFrame([[cm["approved_repaid"], cm["declined_repaid"]], [cm["approved_defaulted"], cm["declined_defaulted"]]],
                   index=["actually repaid", "actually defaulted"], columns=["approved", "declined"]))
print()
pd.Series({k: v for k, v in test_metrics.items() if not isinstance(v, dict)}).round(4)

### 4.5 Scoring applicants (Step 5: `src/predict.py`)
This is the interface the app uses. `load_artifacts()` loads the calibrated model and its metadata once (the app caches it with `st.cache_resource`). `predict()` scores one applicant from a dict of raw fields; `predict_batch()` scores a whole table in one vectorised call. Both use the cut-off and refer band saved in `models/metadata.json`, so the app can't drift from what was trained. Protected attributes are not needed: the applicant below is scored without SEX, MARRIAGE or AGE.

`reasons` and `shap` are empty until the explainability step (section 5) is built.

In [ ]:
from src.predict import load_artifacts, predict, predict_batch

artifacts = load_artifacts()

applicant = test.drop(columns=[TARGET, *PROTECTED]).iloc[0].to_dict()
print("Applicant:", applicant)
predict(applicant, artifacts)

In [ ]:
# A missing field gives a clear error instead of a crash deep inside the model
incomplete = {k: v for k, v in applicant.items() if k != "PAY_0"}
try:
    predict(incomplete, artifacts)
except ValueError as err:
    print("ValueError:", err)

In [ ]:
# Score 1,000 test applicants at once and check the decision mix
scored = predict_batch(test.drop(columns=TARGET).head(1000), artifacts)
print(scored["decision"].value_counts(normalize=True).round(3))
scored[["LIMIT_BAL", "PAY_0", "pd", "decision"]].head()

## 5. Explainability (SHAP) and fairness
_To do: see `docs/BUILD_PLAN.md`._